# CS 9833 Term Project
## James Nicholls, Brodey Miller

In [1]:
import csv
import glob
import sqlite3
import os
import sys
import numpy as np
import pandas as pd

# 1. Prepare dataset

CICIds_2023, MERGED_CSV

## 1.1. Convert the dataset to SQLite database

In [2]:
db_path = "./data/merged_data.sqlite3"

if os.path.exists(db_path):
    print(f"Skipping conversion; {db_path} already exists.")
else:
    """Convert multiple CSV files to a single SQLite database by merging all data into one table."""
    # Get all CSV files matching the pattern
    csv_files = sorted(glob.glob("./data/MERGED_CSV/Merged*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found matching pattern: ./data/MERGED_CSV/Merged*.csv")

    print(f"Found {len(csv_files)} CSV files")

    # Connect to SQLite database (creates if doesn't exist)
    conn = sqlite3.connect("./data/merged_data.sqlite3")
    cursor = conn.cursor()

    # Process files to create a single merged table
    all_rows = []
    all_headers = None
    total_rows = 0

    # Process each CSV file
    for i, csv_file in enumerate(csv_files, 1):
        print(f"[{i}/{len(csv_files)}] Processing {os.path.basename(csv_file)}...")
        
        # Read the CSV file to get headers and data
        with open(csv_file, 'r', newline='') as f:
            reader = csv.reader(f)
            try:
                headers = next(reader)
            except StopIteration:
                print(f"  -> Warning: Empty CSV file {os.path.basename(csv_file)}, skipping")
                continue
            
            # Sanitize column names
            sanitized_headers = []
            for header in headers:
                # Remove spaces, hyphens, and other special chars
                col_name = header.strip().replace(' ', '_').replace('-', '_').replace('.', '_')
                # Remove any non-alphanumeric characters except underscores
                col_name = ''.join(c for c in col_name if c.isalnum() or c == '_')
                # Ensure it starts with a letter
                if col_name and col_name[0].isdigit():
                    col_name = 'col_' + col_name
                sanitized_headers.append(col_name)
            
            # Add a source column to track which CSV the row came from
            if "source_file" not in sanitized_headers:
                sanitized_headers.append("source_file")
            
            # If this is the first file, set up the target table
            if all_headers is None:
                all_headers = sanitized_headers
                
                # Build CREATE TABLE statement for the single merged table
                create_stmt = f"CREATE TABLE IF NOT EXISTS merged_data (id INTEGER PRIMARY KEY AUTOINCREMENT"
                for header in all_headers:
                    create_stmt += f", {header} TEXT"
                create_stmt += ")"
                
                cursor.execute(create_stmt)
            else:
                # Check if headers match - if not, pad with None for missing columns or add extra columns
                if len(sanitized_headers) != len(all_headers):
                    print(f"  -> Warning: Header mismatch! Expected {len(all_headers)} columns, got {len(sanitized_headers)}")
                    # For simplicity, we'll assume all Merged CSV files have the same structure
                    # If headers don't match exactly, we'll use the first file's structure
                    print(f"  -> Using structure from first file (Merged01)")
            
            # Read and insert all rows
            for row in reader:
                total_rows += 1
                
                # Ensure row has correct number of columns
                if len(row) < len(all_headers):
                    row.extend([''] * (len(all_headers) - len(row)))
                elif len(row) > len(all_headers):
                    raise ValueError(
                        f"{os.path.basename(csv_file)} row has "
                        f"{len(row)} columns; expected {len(all_headers)}"
                    )

                # Convert empty strings to None for NULL
                processed_row = []
                for j, cell in enumerate(row):
                    if j < len(all_headers) - 1:  # Don't include source_file in the cell processing
                        if cell == '':
                            processed_row.append(None)
                        else:
                            processed_row.append(cell)
                
                # Add source file identifier (relative path)
                source_file = os.path.basename(csv_file)
                if len(processed_row) == len(all_headers) - 1:
                    processed_row.append(source_file)
                else:
                    # If we have extra columns, add source_file at the end
                    processed_row.extend([''] * (len(all_headers) - len(processed_row) - 1))
                    processed_row.append(source_file)
                
                all_rows.append(tuple(processed_row))
                
                # Batch insert every 1000 rows to improve performance
                if len(all_rows) >= 1000:
                    cursor.executemany(
                        f"INSERT INTO merged_data ({', '.join(all_headers)}) VALUES ({', '.join(['?' for _ in all_headers])})",
                        all_rows
                    )
                    all_rows = []

    # Insert any remaining rows
    if all_rows:
        cursor.executemany(
            f"INSERT INTO merged_data ({', '.join(all_headers)}) VALUES ({', '.join(['?' for _ in all_headers])})",
            all_rows
        )

    # Commit changes and close connection
    conn.commit()
    conn.close()

    print(f"\n✓ Successfully created/merged database: ./data/merged_data.sqlite3")
    print(f"✓ Processed {len(csv_files)} files")
    print(f"✓ Total rows inserted: {total_rows}")
    print(f"✓ Created single table: merged_data with {len(all_headers) if all_headers else 0} columns")

Skipping conversion; ./data/merged_data.sqlite3 already exists.


## 1.2. Import as numpy array

with sqlite3.connect("./data/merged_data.sqlite3") as conn:
    rows = conn.execute("SELECT * FROM merged_data").fetchall()
    
dataset = np.asarray(rows)

## 1.3 Check for null values and impute them

## 1.1. Import as pd dataframe

In [ ]:
csv_files = sorted(glob.glob("./data/MERGED_CSV/Merged*.csv"))
df = pd.DataFrame()
for f in csv_files:
    print(f"Working on {f}")
    current_data = pd.read_csv(f)
    if len(df) == 0:
        df = current_data
    else:
        df.merge(current_data, on="id", how="left")
        
len(df)

Working on ./data/MERGED_CSV/Merged01.csv
Working on ./data/MERGED_CSV/Merged02.csv
Working on ./data/MERGED_CSV/Merged03.csv
Working on ./data/MERGED_CSV/Merged04.csv
Working on ./data/MERGED_CSV/Merged05.csv
Working on ./data/MERGED_CSV/Merged06.csv
Working on ./data/MERGED_CSV/Merged07.csv
Working on ./data/MERGED_CSV/Merged08.csv
Working on ./data/MERGED_CSV/Merged09.csv
Working on ./data/MERGED_CSV/Merged10.csv
Working on ./data/MERGED_CSV/Merged11.csv
Working on ./data/MERGED_CSV/Merged12.csv
Working on ./data/MERGED_CSV/Merged13.csv
Working on ./data/MERGED_CSV/Merged14.csv
Working on ./data/MERGED_CSV/Merged15.csv
Working on ./data/MERGED_CSV/Merged16.csv
Working on ./data/MERGED_CSV/Merged17.csv
Working on ./data/MERGED_CSV/Merged18.csv
Working on ./data/MERGED_CSV/Merged19.csv
Working on ./data/MERGED_CSV/Merged20.csv
Working on ./data/MERGED_CSV/Merged21.csv
Working on ./data/MERGED_CSV/Merged22.csv
Working on ./data/MERGED_CSV/Merged23.csv
Working on ./data/MERGED_CSV/Merge

712311